In [ ]:
import torch  
from transformers import AutoModelForCausalLM, AutoTokenizer  

tokenizer = AutoTokenizer.from_pretrained("state-spaces/mamba-130m-hf")
model = AutoModelForCausalLM.from_pretrained("state-spaces/mamba-130m-hf", dtype=torch.float32, device_map="auto",)  
model.eval()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
prompt = "The capital of France is"

inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs, use_cache=True)

original_cache = outputs.cache_params

In [ ]:
print(type(original_cache))
print(len(original_cache.ssm_states))
print(original_cache.ssm_states[0].shape)

In [ ]:
import torch.nn as nn

state_shape = original_cache.ssm_states[0].shape
B, D, N = state_shape

flatten_dim = D * N
latent_dim = 512

class StateAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return z, x_hat

ae = StateAutoencoder(flatten_dim, latent_dim).to(device)

In [ ]:
state_tensor = original_cache.ssm_states[0]
state_flat = state_tensor.reshape(B, -1)

In [ ]:
ae = ae.to(device=device, dtype=state_flat.dtype)
with torch.no_grad():
    z, reconstructed_flat = ae(state_flat)

reconstructed_state = reconstructed_flat.reshape(B, D, N)

In [ ]:
import copy

reconstructed_cache = copy.deepcopy(original_cache)
reconstructed_cache.ssm_states[0] = reconstructed_state

In [ ]:
with torch.no_grad():
    baseline_out = model.generate(
        **inputs,
        max_new_tokens=20
    )

print("Baseline:")
print(tokenizer.decode(baseline_out[0]))

In [ ]:
new_text = " and it is famous for"

new_inputs = tokenizer(new_text, return_tensors="pt").to(device)

with torch.no_grad():
    resumed_outputs = model.generate(
        **new_inputs,
        max_new_tokens=20,
        cache_params=reconstructed_cache,
        use_cache=True
    )

print("Resumed:")
print(tokenizer.decode(resumed_outputs[0]))

In [ ]:
optimizer = torch.optim.Adam(ae.parameters(), lr=1e-4)
loss_fn = nn.MSELoss()

for step in range(500):
    z, reconstructed_flat = ae(state_flat)
    loss = loss_fn(reconstructed_flat, state_flat)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        print(step, loss.item())